# Single-class object of interest — RT-DETR-L + segmentation (Dice)

Goal: detect and segment **any object of interest** as one class. Original SKU ids are ignored (`nc: 1`, YOLO id `0` / `object`). You will classify crops later.

**Ultralytics RT-DETR-L is detection-only** — it has no mask head and no Dice mask loss. This notebook therefore:

1. Uses the local copy of the Drive dataset under `train_v2/dataset` (single-class polygons).
2. Trains **YOLO11l-seg** with **pure Dice** mask loss so you get boxes + masks.
3. Optionally trains **RT-DETR-L** on the same data for box-only detection.
4. Uses **label-aware** augs (no mosaic / mixup / copy-paste / perspective) so polygons stay in-frame.

Do **not** run the download from this notebook. Use the script below first.

## 0. Download the dataset (run this in a terminal first)

The script copies the previous Drive dataset (`utils.gdrive.drive_folder("dataset")`) into `train_v2/dataset`, shows tqdm byte progress, remaps every class id to `0`, and writes `data.yaml`.

Google Drive for Desktop must be running so `G:\My Drive` (or the Shared Drive shortcut) is visible.

From the **repo root** in PowerShell:

```powershell
python train_v2/utils/download_dataset.py
```

If `train_v2/dataset` already exists and you want a fresh copy:

```powershell
python train_v2/utils/download_dataset.py --force
```

Optional overrides:

```powershell
python train_v2/utils/download_dataset.py --source "G:\path\to\dataset" --dest train_v2/dataset
```

When it finishes you should have:

- `train_v2/dataset/images/{train,val,test}/`
- `train_v2/dataset/labels/{train,val,test}/`  (polygons, class `0`)
- `train_v2/dataset/data.yaml`  (`nc: 1`, `names: {0: object}`)

## 1. Setup and environment check

In [1]:
from pathlib import Path
import sys

import torch
import ultralytics
from ultralytics import YOLO, RTDETR
from ultralytics.utils import SETTINGS

# Skip TensorBoard. This env has NumPy 2 + an old TensorFlow that crash on import.
SETTINGS["tensorboard"] = False

HERE = Path.cwd().resolve()
if not (HERE / "utils" / "download_dataset.py").exists():
    HERE = HERE / "train_v2"

sys.path.insert(0, str(HERE / "utils"))
from augment import LABEL_AWARE_AUG
from dice_loss import apply_dice_mask_loss

DATA_YAML = HERE / "dataset" / "data.yaml"
RUNS = HERE / "runs"

print("=" * 70)
print("TRAIN_V2 — SINGLE-CLASS SEGMENTATION")
print("=" * 70)
print(f"Ultralytics : {ultralytics.__version__}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
print()
print(f"train_v2    : {HERE}")
print(f"data.yaml   : {DATA_YAML}")
print(f"yaml exists : {DATA_YAML.exists()}")
print(f"runs        : {RUNS}")
if not DATA_YAML.exists():
    raise FileNotFoundError(
        "Dataset not found. From the repo root run:\n"
        "  python train_v2/utils/download_dataset.py"
    )
print("=" * 70)

c:\Users\Haqkiem\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


TRAIN_V2 — SINGLE-CLASS SEGMENTATION
Ultralytics : 8.4.146
PyTorch     : 2.11.0+cu128
CUDA        : True
GPU         : NVIDIA GeForce RTX 4050 Laptop GPU

train_v2    : C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2
data.yaml   : C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\dataset\data.yaml
yaml exists : True
runs        : C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\runs


## 2. Confirm the local set is single-class

Every polygon should already be class `0` after the download script. This cell only checks; it does not rewrite labels.

In [2]:
import yaml

cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
print("nc   :", cfg.get("nc"))
print("names:", cfg.get("names"))

class_ids = set()
n_objects = 0
label_files = list((HERE / "dataset" / "labels").rglob("*.txt"))
for path in label_files:
    for line in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        class_ids.add(int(float(parts[0])))
        n_objects += 1

print(f"label files : {len(label_files)}")
print(f"objects     : {n_objects}")
print(f"class ids   : {sorted(class_ids)}")
if class_ids != {0}:
    raise ValueError(
        f"Expected only class 0 after download remap, got {sorted(class_ids)}. "
        "Re-run: python train_v2/utils/download_dataset.py"
    )
print("OK — single class `object` (id 0). SKU labels are ignored.")

nc   : 1
names: {0: 'object'}
label files : 2726
objects     : 6067
class ids   : [0]
OK — single class `object` (id 0). SKU labels are ignored.


## 3. Switch mask loss to Dice

YOLO-seg default mask loss is BCE. This patches `v8SegmentationLoss.single_mask_loss` to **pure Dice** (same autograd-safe crop as `training_dice.ipynb`). Box, classification, and DFL stay Ultralytics defaults.

In [3]:
apply_dice_mask_loss()

Mask loss: pure Dice (box / cls / DFL unchanged)


## 4. Label-aware augmentation

Mosaic, mixup, copy-paste, perspective, shear, and large translate/scale can push a polygon off the image or crop it away. `LABEL_AWARE_AUG` keeps:

- photometric augs (HSV) that do not move the ROI
- horizontal flip
- small rotation (`±8°`), translate (`5%`), scale (`10%`)

`cls` is also lowered because there is only one class — the job is localize + mask, not SKU ID.

In [4]:
print("Label-aware train() kwargs:")
for key, value in LABEL_AWARE_AUG.items():
    print(f"  {key:14s} {value}")

Label-aware train() kwargs:
  mosaic         0.0
  mixup          0.0
  copy_paste     0.0
  cutmix         0.0
  perspective    0.0
  shear          0.0
  translate      0.05
  scale          0.1
  degrees        8.0
  fliplr         0.5
  flipud         0.0
  hsv_h          0.015
  hsv_s          0.5
  hsv_v          0.4
  erasing        0.0
  multi_scale    0.0
  close_mosaic   0


## 5. Train YOLO11l-seg (boxes + masks, Dice)

This is the run that actually **detects and segments**. Weights land in `train_v2/runs/yolo11l_seg_object_dice/`.

If you OOM on the GPU, drop `batch` from `4` to `2`.

In [5]:
seg_model = YOLO("yolo11l-seg.pt")

seg_results = seg_model.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=4,
    patience=20,
    device=0 if torch.cuda.is_available() else "cpu",
    project=str(RUNS),
    name="yolo11l_seg_object_dice",
    exist_ok=True,
    cls=0.1,
    **LABEL_AWARE_AUG,
)

print(seg_results)

New https://pypi.org/project/ultralytics/8.4.149 available  Update with 'pip install -U ultralytics'


c:\Users\Haqkiem\AppData\Local\Programs\Python\Python310\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Ultralytics 8.4.146  Python-3.10.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=0, cls=0.1, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\dataset\data.yaml, degrees=8.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.0, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11l-seg.pt, mome

2026/09/12 14:22:48 INFO mlflow.tracking.fluent: Experiment with name 'C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\runs' does not exist. Creating a new experiment.


MLflow: logging run_id(8fc512b6ba5d4af2aacca93355c36a12) to runs\mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs\mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Using 1909 train, 403 val images for fraction=1.0 at imgsz=640
Using 8 dataloader workers
Logging results to C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\runs\yolo11l_seg_object_dice
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      1/100      3.72G      1.951      1.544     0.5835      2.126          0          2        640: 100% ━━━━━━━━━━━━ 478/478 2.3it/s 3:300.6ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 51/51 1.5it/s 33.7s0.4ss
                   all        403        811    0.00744      0.234    0.00333   0.000928    0.00516      0.158 

## 6. Optional — RT-DETR-L detection only

Ultralytics `RTDETR("rtdetr-l.pt")` trains **boxes**, not masks. Polygons in the YAML set are converted to xyxy internally. Skip this cell if you only need the segmentor above.

RT-DETR is heavier than YOLO11l-seg; start at `batch=2` if memory is tight.

In [ ]:
# Optional. Comment this cell out if you only want masks from section 5.
det_model = RTDETR("rtdetr-l.pt")

det_results = det_model.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=2,
    patience=20,
    device=0 if torch.cuda.is_available() else "cpu",
    project=str(RUNS),
    name="rtdetr_l_object_detect",
    exist_ok=True,
    cls=0.1,
    **LABEL_AWARE_AUG,
)

print(det_results)

## 7. Quick sanity check on a val image

Loads `best.pt` from the segmentation run and draws boxes + masks. Classification later can consume these crops.

In [6]:
best = RUNS / "yolo11l_seg_object_dice" / "weights" / "best.pt"
if not best.exists():
    raise FileNotFoundError(f"No weights yet: {best}")

val_images = sorted(
    p for p in (HERE / "dataset" / "images" / "val").iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
sample = val_images[0]
print("Sample:", sample)

pred_model = YOLO(str(best))
results = pred_model.predict(source=str(sample), conf=0.25, verbose=False)[0]
print(f"detections: {len(results.boxes)}  (class ignored — all are `object`)")
results.save(filename=str(HERE / "preview_seg.jpg"))
print("Wrote", HERE / "preview_seg.jpg")

Sample: C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\dataset\images\val\C001_tracing_paper__C001_TRACKING PAPER_C001_05.jpg
detections: 1  (class ignored — all are `object`)
Wrote C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\preview_seg.jpg
